# 9. Generalized Reynolds Equation

The Reynolds equation with Elrod-Adams cavitation, written with two generalized coefficients:

$$\frac{\partial}{\partial t}\big(\rho_\mathrm{eff} \, h (1-\theta)\big) = \nabla \cdot \left(\varepsilon \, \nabla p\right) - \nabla \cdot \left(\rho_\mathrm{eff} \, h (1-\theta) \, \mathbf{U}_m\right), \qquad \mathbf{U}_m = \tfrac{1}{2}\left(\mathbf{U}_\mathrm{bot} + \mathbf{U}_\mathrm{top}\right)$$

- Poiseuille coefficient $\varepsilon(p, h)$, default $\varepsilon = \dfrac{\rho h^3}{12 \eta}$
- effective, height-averaged density $\rho_\mathrm{eff}(p, h)$, default $\rho_\mathrm{eff} = \rho$

Note that $\rho_\mathrm{eff}$ enters both the time derivative and the Couette term. The generalized form is activated in the YAML file by

```yaml
implicit_solver:
    equations:
        reynolds: True
        cavitation: True
        generalized_reynolds: True
```

## Customized Coefficients

The coefficients are replaced by <code style="color:#c586c0">problem.set_custom_epsilon(function)</code> and <code style="color:#c586c0">problem.set_custom_rho_eff(function)</code>. Both functions take the pressure $p$ [Pa] and the gap height $h$ [m] as scalars and must be JAX-differentiable, since $\partial \varepsilon / \partial p$, $\partial \varepsilon / \partial h$ and $\partial \rho_\mathrm{eff} / \partial p$, $\partial \rho_\mathrm{eff} / \partial h$ enter the Newton Jacobian.

Density and viscosity are not passed, so the function has to contain the equation of state and the viscosity law in closed form:

$$p \;\xrightarrow{\text{EoS}^{-1}}\; \rho, \qquad p \;\xrightarrow{\text{Barus}}\; \eta, \qquad \varepsilon = \frac{\rho(p) \, h^3}{12 \, \eta(p)}$$

For demonstration, we now look at a parabolic slider with cavitation, Dowson-Higginson EoS and Barus piezoviscosity. Three cases are compared:

- classical Reynolds implementation
- generalized Reynolds, default coefficients
- generalized Reynolds, via user-specified functions as shown below

$\rightarrow$ **[YAML File](../../config_lib/validation/parabolic_slider_cav_rey.yaml)**

In [ ]:
import jax.numpy as jnp
from GaPFlow.problem import Problem
from utils import load_template
from utils.solver_comparison import build_config
from utils.plotting import plot_custom_eos_comparison

P0, rho0, C1, C2 = 101325., 877.7007, 2.22e9, 1.66
eta0, aB = 0.039, 3.0e-9

# user-specified functions for generalized Reynolds equation
def rho_dh(p):
    return rho0 * (C2 * (p - P0) + C1) / ((p - P0) + C1)


def barus(p):
    return eta0 * jnp.exp(aB * p)


def epsilon(p, h):
    return rho_dh(p) * h ** 3 / (12.0 * barus(p))


def rho_eff(p, h):
    return rho_dh(p)


template = load_template('parabolic_slider_cav_rey')
overrides = {
    'options.silent': True,
    'grid.Nx': 100,
    'numerics.max_it': 300,
    'properties.piezo': {'name': 'Barus', 'aB': aB},
    'implicit_solver.equations.reynolds': True,
}

results = {}
for label, generalized, custom in [('Reynolds', False, False),
                                   ('generalized, default', True, False),
                                   ('generalized, custom', True, True)]:
    config = build_config(template, 'implicit', **overrides,
                          **{'implicit_solver.equations.generalized_reynolds': generalized})
    problem = Problem.from_string(config)
    if custom:  # here we set the custom functions for epsilon and rho_eff
        problem.set_custom_epsilon(epsilon)
        problem.set_custom_rho_eff(rho_eff)
    problem.run()

    nf = problem.solver.quad_mgr.nf
    X = problem.topo.xx[1:-1, 1] / problem.grid['Lx']
    rho_cav = (problem.q[0] * (1.0 - nf('theta')))[1:-1, 1]
    results[label] = (X, nf('p')[1:-1, 1].copy(), rho_cav.copy())

plot_custom_eos_comparison(results)

In [ ]:
import numpy as np

p_ref = results['Reynolds'][1]
for label in ['generalized, default', 'generalized, custom']:
    dp = np.max(np.abs(results[label][1] - p_ref)) / np.max(p_ref)
    print(f"{label:<22} max |p - p_Reynolds| / max p = {dp:.1e}")

## Elastic Deformation: Effective Influence Newton Method (EINM)

Here, we look at a short demonstration / validation of how important it is to include information of the elastic deformation into the system matrix for the fluid update.

With an elastic surface, the gap height depends on the pressure, $h = h(p)$. **Without EINM**, the deformation is lagged in Picard-style iteration: $h$ is computed from the previous iterate and underrelaxed with <code>properties.elastic.alpha_underrelax</code> $= \alpha$.

**With EINM** (<code>fem_solver.physics.einm: true</code>), the sensitivity $\partial h / \partial p$ (local $3 \times 3$ elastic Green's function) is folded into the Newton Jacobian via $\partial R / \partial h$ of each term, so no underrelaxation is needed ($\alpha = 1$).

We now look at the following use case to validate the EINM method: Two flat plates (no sliding, periodic domain, DH fluid, $p_0 = 1 \text{bar}$) are squeezed by 5 % at $t = 0$. The script below runs this problem with different settings and plots the convergence of $p$ and $h$ over the Newton iterations:
- EINM activated, $\alpha = 1$
- EINM not activated, with different $\alpha$ values

**The problem**: The squeeze leads to a pressure increase, which in turn causes deformation, which in turn widens the gap again, which in turn decreases the pressure...

$\rightarrow$ **[YAML File](../../config_lib/validation/squeeze_elastic_gen_reynolds.yaml)**

In [ ]:
import numpy as np
from GaPFlow.problem import Problem
from utils.plotting import plot_einm_convergence

DH0 = -0.05e-6  # sudden squeeze step at t = 0 [m]


def run_case(einm, alpha):
    problem = Problem.from_yaml('../../config_lib/validation/squeeze_elastic_gen_reynolds.yaml')
    problem.fem_solver['physics']['einm'] = einm
    problem.topo.ElasticDeformation.alpha_underrelax = alpha

    hist = {'p': [], 'h': []}

    def record(solver):
        hist['p'].append(solver.quad_mgr.nf('p')[1:-1, 1:-1].mean())
        hist['h'].append(solver.problem.topo.h[1:-1, 1:-1].mean())

    problem.solver.cb_list.append(record)
    problem._pre_run()
    problem.topo.h0 = DH0  # here we apply the rigid height change (squeeze)
    problem.run()
    return {key: np.array(val) for key, val in hist.items()}


hist_einm = run_case(einm=True, alpha=1.0)
hist_picard = {alpha: run_case(einm=False, alpha=alpha) for alpha in [0.2, 0.3, 0.4]}

plot_einm_convergence(hist_einm, hist_picard, n_iter=10)

Here, you can see clearly that including the elastic deformation into the system matrix stabilizes the convergence. For this simplified case with periodic boundary conditions and a uniform domain, we obtain a very good linear approximation - despite the cutoff of the elastic deformation information - so that convergence is obtained within one Newton iteration. Note that for more complex systems, the approximation accuracy of the EINM linearization decreases and more iterations are required for convergence.

In the cases without EINM activated, a high pressure increase is computed in the first iteration due to the (rigid) squeeze. Only after the fluid is updated, the deformation is computed. Due to the low compressibility of the fluid, the pressure response is very strong. For $\alpha = 0.4$, the elastic deformation after the first fluid update is so high, that the pressure drops to zero (cavitation occurs) in the second fluid update. When reducing underrelaxation, e.g. $\alpha = 0.5$, you can observe that convergence is no longer obtained.

## Elastohydrodynamic Line Contact

A cylinder ($R_x = 15$ mm) slides on a plane with $U = 2$ m/s under a load of $F = 0.1$ MN/m (line contact). Both bodies are elastic ($E = 210$ GPa, $\nu = 0.3$). The rigid-body gap $h_0$ is adjusted by a force balance until the film carries the target load. The equivalent dry (Hertz) contact has the half-width $a = 128.7\,\mu \text{m}$ and a maximum pressure of $p_h = 0.49$ GPa. The lubricant follows the Dowson-Higginson EoS and the Roelands piezoviscosity law.

In the following, we approach this problem in the *naive (slow)* way by starting with a large separation $h_{\text{min}}=10\ \mu \text{m}$, and then iteratively decreasing the distance of cylinder and plane until we reach target force in the film.

The results are shown in dimensionless form, $X = x/a$, $P = p/p_h$ and $H = h R_x/a^2$. Note that reference data is not accurate (scanned from book and digitized). The full run takes around 1-2 minutes.

$\rightarrow$ **[YAML File](../../config_lib/validation/line_contact.yaml)**

In [ ]:
import jax.numpy as jnp
from GaPFlow.problem import Problem
from utils.plotting import plot_line_contact

rho0, P0, C1, C2 = 877.7007, 0., 5.8823529411764706e8, 1.3529411764705883
eta0, eta_inf, p_R, z = 12.0e-3, 6.31e-5, 1.96e8, 0.5605282300320842


def rho_dh(p):
    return rho0 * (C2 * (p - P0) + C1) / ((p - P0) + C1)


def roelands(p):
    return eta0 * jnp.exp(jnp.log(eta0 / eta_inf) * (-1.0 + (1.0 + p / p_R) ** z))


def epsilon(p, h):
    return rho_dh(p) * h ** 3 / (12.0 * roelands(p))


def rho_eff(p, h):
    return rho_dh(p)


problem = Problem.from_yaml('../../config_lib/validation/line_contact.yaml')
problem.set_custom_epsilon(epsilon)
problem.set_custom_rho_eff(rho_eff)

# first settle the film, then adjust h0 until the film carries the load
problem.topo.force_balance = False


def control():
    if problem.step == 10:
        problem.topo.force_balance = True
    if problem.topo.force_balance and problem.topo._fb_controller.controller.converged:
        problem._stop = True


problem.add_callback(control)
problem.run()

plot_line_contact(problem)

See below for an animation of the convergence. Note that we basically solve the problem for a high number of different target loads. For more efficient runs, you would normally start with the Hertzian dry contact solution, i.e. use Hertzian pressure and deformation as initial guess together with a $h_\text{min}$ guess e.g. from Dowson-Hamrock. 

In [ ]:
problem.animate()